# NIRA case study on Google Colab

Buuggan wuxuu ka furaa dashboard-ka natiijooyinka horey loo kaydiyay. Moodellada dib looma tababaro.

This notebook opens the research dashboard from the saved outputs. It does not refit models.

## Before you run

Open this notebook with the Colab button in the GitHub README, then run the cells from top to bottom. The first cell clones the public repository when the saved outputs are not already in the session.

The raw comment file and the annotation workbook are not in the public repository, because some comments contain phone numbers. The dashboard uses the saved result tables.


In [ ]:
from pathlib import Path
import os
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False
DRIVE_ZIP = Path("/content/drive/MyDrive/NIRA_CASE_STUDY.zip")


def find_root(start):
    start = Path(start)
    candidates = []
    if start.exists():
        candidates.append(start)
        candidates.extend(start.parents)
        if start.is_dir():
            candidates.extend(path for path in start.iterdir() if path.is_dir())
    for candidate in candidates:
        app = candidate / "dashboard" / "streamlit_app.py"
        saved = candidate / "outputs" / "06_results_predictions" / "20261019_predictions" / "corpus_predictions.csv"
        if app.exists() and saved.exists():
            return candidate.resolve()
    return None


ROOT = find_root(Path.cwd())
if ROOT is None and Path("/content").exists():
    ROOT = find_root(Path("/content"))

if ROOT is None and IN_COLAB:
    repo = "https://github.com/Raggeh/NIRA_CASE_STUDY.git"
    target = Path("/content/NIRA_CASE_STUDY")
    if not target.exists():
        subprocess_import = __import__("subprocess")
        subprocess_import.check_call(["git", "clone", "--depth", "1", repo, str(target)])
    ROOT = find_root(target)

if ROOT is None and IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if DRIVE_ZIP.exists():
        zipfile.ZipFile(DRIVE_ZIP).extractall("/content")
        ROOT = find_root(Path("/content"))

if ROOT is None and IN_COLAB:
    from google.colab import files
    print("Upload NIRA_CASE_STUDY.zip")
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(name).extractall("/content")
    ROOT = find_root(Path("/content"))

if ROOT is None:
    raise FileNotFoundError(
        "Project folder not found. Upload NIRA_CASE_STUDY.zip, or open this notebook from that folder."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT)
print("Raw comments file:", (ROOT / "NIRA_Facebook_Datasets_5292.csv").exists())


## Install the dashboard packages

These packages only read the saved tables. The embedding and topic-model packages are not installed here.


In [ ]:
import subprocess
import sys

packages = [
    "streamlit==1.66.0",
    "pandas==3.0.5",
    "numpy==2.5.1",
    "plotly==7.1.0",
    "openpyxl==3.1.5",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dashboard packages installed.")


## Check the saved results

The dashboard reads these files. A missing file means the zip did not include the `outputs` folder.


In [ ]:
from pathlib import Path
import pandas as pd

required = [
    "Publication Date Verification.xlsx",
    "outputs/02_eda/20261018_eda/eda_findings.json",
    "outputs/04_topic_modeling/20261019_topic_models_function_words/topic_model_selection.json",
    "outputs/05_model_development/20261019_supervised/selected_models.json",
    "outputs/06_results_predictions/20261019_predictions/corpus_predictions.csv",
    "outputs/06_results_predictions/20261019_predictions/comment_comparison_sample.csv",
    "outputs/07_events_timeline/20261019_events/figures/feedback_loop_sequence.png",
    "dashboard/streamlit_app.py",
]
missing = [name for name in required if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Missing from the project folder:\n" + "\n".join(missing))

predictions = pd.read_csv(ROOT / "outputs/06_results_predictions/20261019_predictions/corpus_predictions.csv")
sample = pd.read_csv(ROOT / "outputs/06_results_predictions/20261019_predictions/comment_comparison_sample.csv")
print("Corpus prediction rows:", len(predictions))
print("Side-by-side comments:", sample["record_key"].nunique())
print("Saved outputs are present.")


## Open the dashboard

On Colab, the cell prints a link. On this computer, open `http://localhost:8501`.


In [ ]:
import subprocess
import sys
import time
import urllib.request

log_path = ROOT / "colab_streamlit.log"
log_file = open(log_path, "w", encoding="utf-8")
command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "dashboard/streamlit_app.py",
    "--server.port",
    "8501",
    "--server.address",
    "0.0.0.0",
    "--server.headless",
    "true",
    "--server.enableCORS",
    "false",
    "--server.enableXsrfProtection",
    "false",
    "--browser.gatherUsageStats",
    "false",
]
streamlit_process = subprocess.Popen(command, cwd=ROOT, stdout=log_file, stderr=subprocess.STDOUT)

started = False
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:8501", timeout=2)
        started = True
        break
    except Exception:
        if streamlit_process.poll() is not None:
            break
        time.sleep(1)

if not started:
    log_file.flush()
    print(log_path.read_text(encoding="utf-8")[-4000:])
    raise RuntimeError("Streamlit did not start. The log above is the reason.")

if IN_COLAB:
    from google.colab.output import eval_js
    dashboard_url = eval_js("google.colab.kernel.proxyPort(8501)")
    print("Dashboard:", dashboard_url)
else:
    print("Dashboard: http://localhost:8501")


## Optional: re-run the analysis notebooks

Leave `RUN_ANALYSIS` as `False`. The cited results are already in `outputs/`.

Setting it to `True` installs the full model stack and runs notebooks 01 through 08. Notebook 09 is not run: regenerating it would replace the hand-edited final report. This step refits the models and can take a long time.


In [ ]:
RUN_ANALYSIS = False

if not RUN_ANALYSIS:
    print("Analysis notebooks were not run. The dashboard is using the saved outputs.")
else:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
    import nbformat
    from nbclient import NotebookClient

    notebooks = [
        "notebooks/01_Data_Preparation.ipynb",
        "notebooks/02_EDA.ipynb",
        "notebooks/03_Annotation.ipynb",
        "notebooks/04_Topic_Modeling.ipynb",
        "notebooks/05_Model_Development.ipynb",
        "notebooks/06_Results_and_Predictions.ipynb",
        "notebooks/07_Events_Timeline.ipynb",
        "notebooks/08_Results_Visualization.ipynb",
    ]
    for name in notebooks:
        path = ROOT / name
        print("Running", name)
        notebook = nbformat.read(path, as_version=4)
        client = NotebookClient(notebook, timeout=7200, kernel_name="python3", resources={"metadata": {"path": str(ROOT)}})
        client.execute()
        nbformat.write(notebook, path)
        print("Finished", name)
    print("Notebook 09 was left unchanged so the edited final report stays in place.")
